In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = pd.read_csv('/content/drive/MyDrive/Data visualization/sales_data.csv')

In [6]:
df

,order_id,order_date,customer_id,customer_name,city,product,category,quantity,unit_price,payment_method
0,1001,2024-01-02,C001,Alice,New York,Laptop,Electronics,1,1200,Credit Card
1,1002,2024-01-02,C002,Bob,Los Angeles,Headphones,Electronics,2,150,PayPal
2,1003,2024-01-03,C003,Charlie,New York,Office Chair,Furniture,1,350,Credit Card
3,1004,2024-01-03,C001,Alice,New York,Mouse,Electronics,3,25,Debit Card
4,1005,2024-01-04,C004,Diana,Chicago,Desk,Furniture,1,500,Bank Transfer
5,1006,2024-01-04,C005,Eve,Chicago,Laptop,Electronics,1,1100,Credit Card
6,1007,2024-01-05,C002,Bob,Los Angeles,Monitor,Electronics,2,300,Debit Card
7,1008,2024-01-05,C003,Charlie,New York,Desk Lamp,Furniture,2,45,PayPal
8,1009,2024-01-06,C006,Frank,Miami,Tablet,Electronics,1,600,Credit Card
9,1010,2024-01-06,C001,Alice,New York,Keyboard,Electronics,1,80,Bank Transfer


In [7]:
df['total_amount'] = df['quantity'] * df['unit_price']

# Tính tổng chi tiêu, số đơn hàng và AOV
customer_stats = df.groupby('customer_id').agg(
    total_spending=('total_amount', 'sum'),
    order_count=('order_id', 'count')
)

customer_stats['average_order_value'] = customer_stats['total_spending'] / customer_stats['order_count']
print(customer_stats.head())

             total_spending  order_count  average_order_value
customer_id                                                  
C001                   1355            3           451.666667
C002                    900            2           450.000000
C003                    440            2           220.000000
C004                    500            1           500.000000
C005                   1100            1          1100.000000


In [8]:
# Tính tổng doanh thu toàn bộ và theo từng danh mục
total_revenue = df['total_amount'].sum()
category_revenue = df.groupby('category')['total_amount'].sum().reset_index()

# Tính % đóng góp
category_revenue = category_revenue.assign(
    percentage_contribution = (category_revenue['total_amount'] / total_revenue) * 100
)
print(category_revenue)

      category  total_amount  percentage_contribution
0  Electronics          3955                80.796731
1    Furniture           940                19.203269


In [9]:
# Tính tổng chi tiêu mỗi khách hàng và sắp xếp giảm dần
customer_spending = df.groupby('customer_id')['total_amount'].sum().sort_values(ascending=False)

# Xác định ngưỡng 20% số lượng khách hàng
top_20_threshold = customer_spending.quantile(0.8)
top_customers = customer_spending[customer_spending >= top_20_threshold]

print(f"Số lượng khách hàng thuộc top 20%: {len(top_customers)}")

Số lượng khách hàng thuộc top 20%: 2


In [10]:
# Thống kê mô tả giá theo danh mục
price_analysis = df.groupby('category')['unit_price'].agg(['mean', 'median', 'std'])
print(price_analysis)

                   mean  median         std
category                                   
Electronics  493.571429   300.0  487.209010
Furniture    298.333333   350.0  231.858434


In [12]:
# Tính doanh thu theo ngày và % tăng trưởng
daily_revenue = df.groupby('order_date')['total_amount'].sum().to_frame()
daily_revenue['growth_rate'] = daily_revenue['total_amount'].pct_change() * 100
print(daily_revenue.head())

            total_amount  growth_rate
order_date                           
2024-01-02          1500          NaN
2024-01-03           425   -71.666667
2024-01-04          1600   276.470588
2024-01-05           690   -56.875000
2024-01-06           680    -1.449275


In [13]:
# Tính trung bình trượt 3 ngày
daily_revenue['rolling_avg_3d'] = daily_revenue['total_amount'].rolling(window=3).mean()
print(daily_revenue[['total_amount', 'rolling_avg_3d']].head())

            total_amount  rolling_avg_3d
order_date                              
2024-01-02          1500             NaN
2024-01-03           425             NaN
2024-01-04          1600          1175.0
2024-01-05           690           905.0
2024-01-06           680           990.0


In [14]:
# Xác định ngưỡng outlier (ví dụ: Mean + 3*Std)
mean_val = df['total_amount'].mean()
std_val = df['total_amount'].std()
threshold = mean_val + (3 * std_val)

outliers = df[df['total_amount'] > threshold]
print(f"Số lượng đơn hàng bất thường: {len(outliers)}")

Số lượng đơn hàng bất thường: 0
